#### From week 3 section 

In [2]:
import pandas as pd
import plotly.express as px


In [3]:
remissions_df = pd.read_csv("../data/processed/remissions_db.csv")
remissions_df.head(-1)

,Year,tkt_code,order_date,order_code,start_time,truck_code,ship_plant_code,u_Volumen,typed_time,at_plant_time,u_Cicle,name,Nombre del proyecto,ship_addr_line,map_page
0,2020,51013232,2020-02-04 00:00:00,1073,2020-02-04 07:00:00,7044,510,6.0,2020-02-04 06:45:13,2020-02-04 08:17:24,92,ONE TIME OCTAVIO RIOS,ARMENDARIZ ARMANDO,HACIENDAS HENEKENERAS 2679 GRACC HACIEND,CH-F1
1,2020,51013233,2020-02-04 00:00:00,1073,2020-02-04 07:00:00,6603,510,5.5,2020-02-04 06:45:22,2020-02-04 08:26:06,101,ONE TIME OCTAVIO RIOS,ARMENDARIZ ARMANDO,HACIENDAS HENEKENERAS 2679 GRACC HACIEND,CH-F1
2,2020,51013235,2020-02-04 00:00:00,1028,2020-02-04 08:00:00,9631,510,2.5,2020-02-04 07:37:56,2020-02-04 08:55:49,78,ONE TIME ESP ANGELICA RIVERA GOMEZ,ROMERO JULIA,CALLE MANUEL BECERRA 14515 COL ALAMEDAS,CH-F5
3,2020,51013236,2020-02-04 00:00:00,1005,2020-02-04 08:15:00,6598,510,3.0,2020-02-04 08:02:36,2020-02-04 09:47:43,105,IVAN NOE SIMENTAL ORTEGA,COLECTOR SACRAMENTO,ARROLLO MIMBRE Y VIALIDAD SACRAMENTO,CH-J9
4,2020,51013238,2020-02-04 00:00:00,1026,2020-02-04 08:30:00,10145,510,3.5,2020-02-04 08:09:01,2020-02-04 09:09:15,60,ONE TIME CONSTRUCENTRO CHIH,MOLINA BALDERRAMA CESAR,PERIF DE LA JUVENTUD 9926 COL RESIDENCIA,CH-L5
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
340768,2026,71037451,2026-01-31 00:00:00,1069,2026-01-31 11:00:00,13053,710,2.0,2026-01-31 10:06:20,2026-01-31 11:03:20,57,BENJAMIN MATA CONDE,VILLAS LA HACIENDITA,EJIDO LA HACIENDITA S/N CHIHUAHUA,CHP-4
340769,2026,71037452,2026-01-31 00:00:00,1073,2026-01-31 10:00:00,12831,710,3.0,2026-01-31 10:36:44,2026-01-31 10:50:38,14,FERRETERIA MOLINA DE CHIHUAHUA,FERRETERIA MOLINA DE CHIHUAHUA,FRACC DOMINIO LOTE 2 MANZANA 10,CHN-3
340770,2026,71037453,2026-01-31 00:00:00,1244,2026-01-31 10:00:00,12831,710,0.5,2026-01-31 10:51:48,2026-01-31 13:01:36,130,FERRETERIA MOLINA DE CHIHUAHUA,FERRETERIA MOLINA DE CHIHUAHUA,FRACC DOMINIO LOTE 2 MANZANA 10,CHN-3
340771,2026,71037454,2026-01-31 00:00:00,1070,2026-01-31 12:00:00,13041,710,2.5,2026-01-31 10:55:54,2026-01-31 12:52:16,117,ALFONSO ORTEGA ANTILLON,FRAC ALCAZARES (MYKONOS),AVE DE LA CANTERA SN ALCAZARES (MYKON,CHP-3


In [4]:
# Based on EDA analysis, it´s okay to drop rows with null values (<2%)
remissions_df = remissions_df[remissions_df['ship_addr_line'].notnull() & remissions_df['map_page'].notnull() & remissions_df['name'].notnull()]
print("Number of rows after dropping null values:", remissions_df.shape[0])

Number of rows after dropping null values: 340244


In [5]:
# Change 'order_date' and 'typed_time' to datetime format
remissions_df['order_date'] = pd.to_datetime(remissions_df['order_date'], errors='raise', format='mixed')
remissions_df['typed_time'] = pd.to_datetime(remissions_df['typed_time'], errors='raise', format='mixed')
remissions_df['start_time'] = pd.to_datetime(remissions_df['start_time'], errors='raise', format='mixed')
remissions_df['at_plant_time'] = pd.to_datetime(remissions_df['at_plant_time'], errors='raise', format='mixed')
remissions_df.info()

<class 'pandas.DataFrame'>
Index: 340244 entries, 0 to 340773
Data columns (total 15 columns):
 #   Column               Non-Null Count   Dtype         
---  ------               --------------   -----         
 0   Year                 340244 non-null  int64         
 1   tkt_code             340244 non-null  int64         
 2   order_date           340244 non-null  datetime64[us]
 3   order_code           340244 non-null  int64         
 4   start_time           340244 non-null  datetime64[us]
 5   truck_code           340244 non-null  int64         
 6   ship_plant_code      340244 non-null  int64         
 7   u_Volumen            340244 non-null  float64       
 8   typed_time           340244 non-null  datetime64[us]
 9   at_plant_time        340244 non-null  datetime64[us]
 10  u_Cicle              340244 non-null  int64         
 11  name                 340244 non-null  str           
 12  Nombre del proyecto  340244 non-null  str           
 13  ship_addr_line       340244 no

In [6]:
# Duplicate check for all time columns
typed_time_duplicates = remissions_df[remissions_df.duplicated(subset=['start_time', 'at_plant_time','truck_code'], keep=False)]
typed_time_duplicates = typed_time_duplicates.sort_values(by='typed_time', ascending=True)
typed_time_duplicates.head(10)

,Year,tkt_code,order_date,order_code,start_time,truck_code,ship_plant_code,u_Volumen,typed_time,at_plant_time,u_Cicle,name,Nombre del proyecto,ship_addr_line,map_page
77566,2021,51038180,2021-08-12,1155,2021-08-12 10:00:00,6163,510,5.0,2021-08-12 10:22:43,2021-08-12 11:54:06,92,PLANEACION INMOBILIARIA DE,QUORUM COMERCIAL,CITADELA SECTOR 49 FRACC 63 QUORUM COME,CH-N1
77567,2021,51038180,2021-08-30,1243,2021-08-12 10:00:00,6163,510,5.0,2021-08-12 10:22:43,2021-08-12 11:54:06,92,PLANEACION INMOBILIARIA DE,QUORUM COMERCIAL,CITADELA SECTOR 49 FRACC 63 QUORUM COME,CH-N1
77575,2021,51038191,2021-08-12,1155,2021-08-12 10:00:00,9430,510,5.0,2021-08-12 11:39:18,2021-08-12 12:32:11,53,PLANEACION INMOBILIARIA DE,QUORUM COMERCIAL,CITADELA SECTOR 49 FRACC 63 QUORUM COME,CH-N1
77576,2021,51038191,2021-08-30,1243,2021-08-12 10:00:00,9430,510,5.0,2021-08-12 11:39:18,2021-08-12 12:32:11,53,PLANEACION INMOBILIARIA DE,QUORUM COMERCIAL,CITADELA SECTOR 49 FRACC 63 QUORUM COME,CH-N1
77579,2021,51038194,2021-08-12,1155,2021-08-12 10:00:00,9431,510,5.5,2021-08-12 12:00:53,2021-08-12 13:22:00,82,PLANEACION INMOBILIARIA DE,QUORUM COMERCIAL,CITADELA SECTOR 49 FRACC 63 QUORUM COME,CH-N1
77580,2021,51038194,2021-08-30,1243,2021-08-12 10:00:00,9431,510,5.5,2021-08-12 12:00:53,2021-08-12 13:22:00,82,PLANEACION INMOBILIARIA DE,QUORUM COMERCIAL,CITADELA SECTOR 49 FRACC 63 QUORUM COME,CH-N1
77582,2021,51038199,2021-08-12,1155,2021-08-12 10:00:00,10145,510,5.5,2021-08-12 12:20:39,2021-08-12 14:02:20,102,PLANEACION INMOBILIARIA DE,QUORUM COMERCIAL,CITADELA SECTOR 49 FRACC 63 QUORUM COME,CH-N1
77583,2021,51038199,2021-08-30,1243,2021-08-12 10:00:00,10145,510,5.5,2021-08-12 12:20:39,2021-08-12 14:02:20,102,PLANEACION INMOBILIARIA DE,QUORUM COMERCIAL,CITADELA SECTOR 49 FRACC 63 QUORUM COME,CH-N1
77743,2021,51038418,2021-08-17,1074,2021-08-17 07:30:00,10157,510,6.5,2021-08-17 07:17:00,2021-08-17 08:27:28,70,MATERIALES INDUSTRIALES DE,SANTA CLARA PONENTE,TABACALERAS YA RROYO EL CALORIENTO S,CH-G2
77744,2021,51038418,2021-08-20,1336,2021-08-17 07:30:00,10157,510,6.5,2021-08-17 07:17:00,2021-08-17 08:27:28,70,MATERIALES INDUSTRIALES DE,SANTA CLARA PONENTE,TABACALERAS YA RROYO EL CALORIENTO S,CH-G2


In [7]:
print("Number of duplicate rows based on start_time, at_plant_time, and truck_code:", typed_time_duplicates.shape[0])

Number of duplicate rows based on start_time, at_plant_time, and truck_code: 24


In [8]:
# It is impossible for many trucks to have the same start_time and at_plant_time
# Therefore, these are duplicates that should be removed, as they are likely to be errors in the data entry process.
remissions_df = remissions_df.drop(typed_time_duplicates.index)
print("Number of rows after dropping duplicates:", remissions_df.shape[0])

Number of rows after dropping duplicates: 340220


We will check u_Cicle to see if there is any order that was never completed.

In [9]:
# Graph the distribution of u_Cicle to see if there are any outliers
fig = px.histogram(remissions_df, x='u_Cicle', nbins=len(remissions_df['u_Cicle'].unique()), title='Distribution of u_Cicle')
fig.show()

In [10]:
# Boxplot of u_Cicle to check for outliers
fig = px.box(remissions_df, y='u_Cicle', title='Boxplot of u_Cicle')
fig.show()

We will check distribution of volume amount

In [11]:
# Calculate counts and percentages for u_Volumen
counts = remissions_df['u_Volumen'].value_counts().sort_index()
percentages = (counts / counts.sum()) * 100

# Histogram of frequency of each unique volume value to check for outliers
x_labels = counts.index.astype(str)  # treat volumes as categorical so bars are wider
fig = px.bar(x=x_labels, y=counts.values, title='Distribution of Volume',
             labels={'x':'u_Volumen','y':'count'}, text=counts.values)
fig.update_traces(
    textposition='outside',
    texttemplate='%{y}',
    customdata=percentages.values,
    hovertemplate='<b>Volume:</b> %{x}<br><b>Count:</b> %{y}<br><b>Percentage:</b> %{customdata:.2f}%<extra></extra>'
)
fig.update_layout(
    bargap=0.1,        # reduce gap between bars
    xaxis_tickangle=45,
    width=1000
)
fig.show()

In [12]:
# Sum all the tkt_code count and u_Volumen related to each order_code and print the orders with highest to lowest total volume and remissions
order_volume = remissions_df.groupby('order_code').agg(
    u_Volumen=('u_Volumen', 'sum'),
    tkt_code_sum=('tkt_code', 'count')
).reset_index()

order_volume = order_volume.sort_values(by='u_Volumen', ascending=False)
print(order_volume.head(10))

     order_code  u_Volumen  tkt_code_sum
0          1000    4753.50          1024
113        1113    4697.00          1279
86         1086    4647.50          1197
70         1070    4532.50          1154
107        1107    4333.50          1159
54         1054    4229.75          1074
7          1007    4170.50           978
128        1128    4039.00          1142
49         1049    4038.50          1051
101        1101    4024.50          1099


In [13]:
# print the count of different map_page values associated with the order_code = 1000
order_1000_map_pages = remissions_df[remissions_df['order_code'] == 1000]['map_page'].nunique()
print("Count of different map_page values associated with order_code 1000:", order_1000_map_pages)

Count of different map_page values associated with order_code 1000: 116


In [14]:
# print the oldest date of the tkt_code and the newest of the order_code = 1000
order_1000_dates = remissions_df[remissions_df['order_code'] == 1000]['order_date']
oldest_date = order_1000_dates.min()
newest_date = order_1000_dates.max()
print("Oldest date of tkt_code for order_code 1000:", oldest_date)
print("Newest date of tkt_code for order_code 1000:", newest_date)

Oldest date of tkt_code for order_code 1000: 2020-02-05 00:00:00
Newest date of tkt_code for order_code 1000: 2026-01-29 00:00:00


In [15]:
# print the count of how many unique order_code are in total
unique_order_codes = remissions_df['order_code'].nunique()
print("Count of unique order_code in total:", unique_order_codes)

Count of unique order_code in total: 778


In [16]:
# Distribution with x values being the order_code unique values and y values being the count of tkt_code for each order_code
order_code_counts = remissions_df['order_code'].value_counts().sort_index()

fig = px.bar(
    x=order_code_counts.index.astype(str),
    y=order_code_counts.values,
    title='Distribution of tkt_code count by order_code',
    labels={'x': 'order_code', 'y': 'tkt_code count'},
    text=order_code_counts.values
)
fig.update_traces(
    textposition='outside',
    texttemplate='%{y}',
    hovertemplate='<b>Order Code:</b> %{x}<br><b>tkt_code Count:</b> %{y}<extra></extra>'
)
fig.update_layout(
    bargap=0.1,
    xaxis_showticklabels=False,
    width=1000
)
fig.show()


In [17]:
# make a boxplot of the previous graph to check for outliers
fig = px.box(order_code_counts.values, title='Boxplot of tkt_code count by order_code')
fig.update_layout(width=800)
fig.show()

In [18]:
# Check distribution of plants where u_Cicle is 1
u_cicle_1 = remissions_df[remissions_df['u_Cicle'] == 1]
plant_counts = u_cicle_1['ship_plant_code'].value_counts()
print(plant_counts)

ship_plant_code
512    4070
515    1651
710    1363
514    1338
511     748
510     316
717     203
Name: count, dtype: int64


In [19]:
#Looks like the tkt_codes can be repeated after years of orders.
remissions_df[remissions_df['tkt_code'].duplicated(keep=False)].sort_values('tkt_code', ascending=True).head(6)


,Year,tkt_code,order_date,order_code,start_time,truck_code,ship_plant_code,u_Volumen,typed_time,at_plant_time,u_Cicle,name,Nombre del proyecto,ship_addr_line,map_page
1,2020,51013233,2020-02-04,1073,2020-02-04 07:00:00,6603,510,5.5,2020-02-04 06:45:22,2020-02-04 08:26:06,101,ONE TIME OCTAVIO RIOS,ARMENDARIZ ARMANDO,HACIENDAS HENEKENERAS 2679 GRACC HACIEND,CH-F1
310446,2025,51013233,2025-08-02,1035,2025-08-02 07:00:00,9430,510,2.0,2025-08-02 06:46:30,2025-08-02 07:46:17,60,ONE TIME PROMOCIONES OCTAVIO RIOS,MARGARITO ROMERO,ING. CARRILLO 16344 COL TRAHUMARA,CHJ-3
310448,2025,51013236,2025-08-02,1201,2025-08-02 07:00:00,7043,510,4.0,2025-08-02 06:54:06,2025-08-02 08:14:23,80,JONATHAN MARQUEZ BACA,OBRAS VARIAS,ARROYO NARAGUA #2225 LOS ARROYOS,CHD-5
3,2020,51013236,2020-02-04,1005,2020-02-04 08:15:00,6598,510,3.0,2020-02-04 08:02:36,2020-02-04 09:47:43,105,IVAN NOE SIMENTAL ORTEGA,COLECTOR SACRAMENTO,ARROLLO MIMBRE Y VIALIDAD SACRAMENTO,CH-J9
310450,2025,51013239,2025-08-02,1090,2025-08-02 08:00:00,9430,510,6.5,2025-08-02 07:46:56,2025-08-02 09:03:00,77,FERRETERIA MOLINA DE CHIHUAHUA,FERRETERIA MOLINA DE CHIHUAHUA,MONTE HIMALAYA 4341 QUINTAS CAROLINA,CHH-8
5,2020,51013239,2020-02-04,1150,2020-02-04 09:00:00,6611,510,6.0,2020-02-04 08:36:24,2020-02-04 09:30:24,54,JULIO ARMANDO HINOJOS ENRIQUEZ,FRAC CALZADA DEL BOSQUE AGH,FRAC CALZADA DEL BOSQUE AGH FRAC CAL,CH-H3


We will now check the consistency of remissions per plant

In [20]:
print(remissions_df['ship_plant_code'].unique())
# The new plant (717) was not used for the past version of this project

[510 511 512 515 710 717 514]


In [21]:
counts = remissions_df['ship_plant_code'].value_counts()
percentages = (counts / counts.sum()) * 100
print(counts, '\t', percentages.round(2)) # percentages
print("total:", counts.sum())

ship_plant_code
512    79487
510    68930
511    58374
515    49380
710    44728
514    37317
717     2004
Name: count, dtype: int64 	 ship_plant_code
512    23.36
510    20.26
511    17.16
515    14.51
710    13.15
514    10.97
717     0.59
Name: count, dtype: float64
total: 340220


In [22]:
# See lowest and highest datetime for each plant
for plant_code in remissions_df['ship_plant_code'].unique():
    plant = remissions_df[remissions_df['ship_plant_code'] == plant_code]
    print(f"Plant {plant_code}:")
    print(f"  Lowest datetime: {plant['start_time'].min()}")
    print(f"  Highest datetime: {plant['start_time'].max()}")

Plant 510:
  Lowest datetime: 2020-02-04 07:00:00
  Highest datetime: 2026-05-28 22:00:00
Plant 511:
  Lowest datetime: 2020-02-04 07:00:00
  Highest datetime: 2026-05-28 22:00:00
Plant 512:
  Lowest datetime: 2020-02-04 05:00:00
  Highest datetime: 2026-05-28 19:00:00
Plant 515:
  Lowest datetime: 2020-02-04 08:30:00
  Highest datetime: 2026-05-28 22:00:00
Plant 710:
  Lowest datetime: 2020-02-04 07:00:00
  Highest datetime: 2026-05-28 17:45:00
Plant 717:
  Lowest datetime: 2020-02-04 07:30:00
  Highest datetime: 2022-02-09 07:00:00
Plant 514:
  Lowest datetime: 2022-07-20 07:00:00
  Highest datetime: 2026-05-28 22:00:00


In [23]:
# Drop plant 717 due to low number of remissions and being inactive for the past 4 years
remissions_df = remissions_df[remissions_df['ship_plant_code'] != 717]
print("Number of rows after dropping plant 717:", remissions_df.shape[0])

# Also plant 514 starts from 2022, but that is no problem as it covers over a 10% of the total remission count

Number of rows after dropping plant 717: 338216


In [29]:
# There are volume outliers based on EDA, so we will find them and drop them
# print the rows in which `u_Volumen` is greater than 7 or less/equal than 0
outliers_volumen = remissions_df[(remissions_df['u_Volumen'] > 7) | (remissions_df['u_Volumen'] <= 0)]
print("Rows with volume outliers:")
print(outliers_volumen.shape[0])

Rows with volume outliers:
0


In order to identify repeated orders, the row would need to have repeated values in the following columns: order_code, order_date, typed_time, truck_code. 

In [24]:
# Look for repeated order_codes

repeated_orders = remissions_df[remissions_df.duplicated(subset=['order_code', 'order_date','typed_time','truck_code'], keep=False)]
repeated_orders.head(-1)

# No repeated orders.

,Year,tkt_code,order_date,order_code,start_time,truck_code,ship_plant_code,u_Volumen,typed_time,at_plant_time,u_Cicle,name,Nombre del proyecto,ship_addr_line,map_page


In [25]:
# Count unique values for each column
unique_counts = remissions_df.nunique()
print(unique_counts)

Year                        7
tkt_code               323954
order_date               1688
order_code                777
start_time              64996
truck_code                149
ship_plant_code             6
u_Volumen                  32
typed_time             334710
at_plant_time          334635
u_Cicle                   209
name                     2019
Nombre del proyecto     22844
ship_addr_line          70753
map_page                  952
dtype: int64


In [26]:
remissions_df.shape[0]

338216

Based on the initial unique values for each column:
- There have been 777 orders in total
- There have been 149 trucks
- There have been 22844 different projects
- There have been 2019 different clients (apparently)
- There have been 70753 different addresses for delivery

#### Exporting Cleaned Dataset

In [27]:
# Has to be xlsx because of datetime format
remissions_df.to_excel("../data/processed/remissions_db_cleaned.xlsx", index=False)